# CUDA Kernel Demo (RTX 3050)

This notebook demonstrates **CUDA kernels** in this repo:
- **Transformer kernels**: fused QKV, softmax, layernorm, GELU, fused MLP (FP16)
- **Custom Conv2D** (optional): 3×3 conv via `custom_conv` extension

**Run from repo root** so imports resolve. Requires PyTorch with CUDA; transformer CUDA kernels JIT-compile on first use (Ninja + Visual Studio on Windows).

In [ ]:
import sys
import time
from pathlib import Path
root = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Algorithm: Transformer CUDA Kernels

- **Fused QKV**: `y = x @ W + b` with `x` (M, H), `W` (H, 3H). Single matmul + bias.
- **Softmax**: Row-wise exp(x - max) / sum(exp); warp-level reduction.
- **LayerNorm**: (x - μ) * rstd * γ + β over last dimension.
- **GELU**: Approximate 0.5*x*(1+tanh(...)).
- **Fused MLP**: linear2(GELU(linear1(x))) in one kernel.

All use **FP16**; accumulation in FP32 where needed.

## Example: Load and run transformer CUDA kernels

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA required")

from benchmarks.transformer_reference import (
    fused_qkv_pytorch,
    softmax_pytorch,
    layernorm_pytorch,
    gelu_pytorch,
    fused_mlp_pytorch,
)

B, S, H = 2, 64, 128
M = B * S
H3, H4 = H * 3, H * 4
dtype = torch.float16

# Reference (PyTorch)
x = torch.randn(M, H, device="cuda", dtype=dtype)
w_qkv = torch.randn(H, H3, device="cuda", dtype=dtype)
b_qkv = torch.randn(H3, device="cuda", dtype=dtype)
y_ref = fused_qkv_pytorch(x, w_qkv, b_qkv)
print("QKV ref shape:", y_ref.shape)

# CUDA (if extension loads)
try:
    from gpu_kernels.transformer.transformer_cuda import fused_qkv_cuda
    y_cuda = fused_qkv_cuda(x, w_qkv, b_qkv)
    if y_cuda is not None:
        err = (y_cuda - y_ref).abs().max().item()
        print(f"QKV CUDA max diff: {err}")
    else:
        print("QKV CUDA: extension not built")
except Exception as e:
    print("QKV CUDA:", e)

## Benchmark: PyTorch vs CUDA (transformer kernels)

In [ ]:
def run_bench(fn, warmup=5, repeat=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeat):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000 / repeat

B, S, H = 8, 256, 768
M = B * S
H3, H4 = H * 3, H * 4
dtype = torch.float16

results = []
# QKV
x = torch.randn(M, H, device="cuda", dtype=dtype)
w = torch.randn(H, H3, device="cuda", dtype=dtype)
b = torch.randn(H3, device="cuda", dtype=dtype)
t_pt = run_bench(lambda: fused_qkv_pytorch(x, w, b))
results.append(("QKV", "PyTorch", t_pt))
try:
    from gpu_kernels.transformer.transformer_cuda import fused_qkv_cuda
    if fused_qkv_cuda(x, w, b) is not None:
        t_cuda = run_bench(lambda: fused_qkv_cuda(x, w, b))
        results.append(("QKV", "CUDA", t_cuda))
except Exception:
    pass

# Softmax
x_s = torch.randn(M, H, device="cuda", dtype=dtype)
t_pt = run_bench(lambda: softmax_pytorch(x_s, dim=-1))
results.append(("Softmax", "PyTorch", t_pt))
try:
    from gpu_kernels.transformer.transformer_cuda import softmax_cuda
    if softmax_cuda(x_s, -1) is not None:
        results.append(("Softmax", "CUDA", run_bench(lambda: softmax_cuda(x_s, -1))))
except Exception:
    pass

# MLP
w1 = torch.randn(H, H4, device="cuda", dtype=dtype)
b1 = torch.randn(H4, device="cuda", dtype=dtype)
w2 = torch.randn(H4, H, device="cuda", dtype=dtype)
b2 = torch.randn(H, device="cuda", dtype=dtype)
t_pt = run_bench(lambda: fused_mlp_pytorch(x, w1, b1, w2, b2))
results.append(("MLP", "PyTorch", t_pt))
try:
    from gpu_kernels.transformer.transformer_cuda import fused_mlp_cuda
    if fused_mlp_cuda(x, w1, b1, w2, b2) is not None:
        results.append(("MLP", "CUDA", run_bench(lambda: fused_mlp_cuda(x, w1, b1, w2, b2))))
except Exception:
    pass

for name, impl, ms in results:
    print(f"  {name} {impl}: {ms:.4f} ms")

## Visualize performance

In [ ]:
import matplotlib.pyplot as plt

kernels = sorted(set(r[0] for r in results))
impls = ["PyTorch", "CUDA"]
data = {impl: [] for impl in impls}
for k in kernels:
    for impl in impls:
        t = next((r[2] for r in results if r[0] == k and r[1] == impl), None)
        data[impl].append(t if t is not None else 0)

x = range(len(kernels))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - w/2 for i in x], data["PyTorch"], w, label="PyTorch", color="#2ecc71")
ax.bar([i + w/2 for i in x], data["CUDA"], w, label="CUDA", color="#3498db")
ax.set_xticks(x)
ax.set_xticklabels(kernels)
ax.set_ylabel("Latency (ms)")
ax.set_title("Transformer kernels: PyTorch vs CUDA (B=8, S=256, H=768)")
ax.legend()
plt.tight_layout()
plt.show()

## Optional: Custom Conv2D (CUDA extension)

If you have built the `custom_conv` extension (`pip install -e extension/`), it supports 3×3 conv with C_in=1, C_out=32, **float32**.

In [ ]:
if torch.cuda.is_available():
    import torch.nn as nn
    B, C_in, H, W = 32, 1, 28, 28
    C_out = 32
    x = torch.randn(B, C_in, H, W, device="cuda", dtype=torch.float32)
    conv = nn.Conv2d(C_in, C_out, 3, padding=0).cuda()
    t_torch = run_bench(lambda: conv(x), repeat=30)
    print(f"torch.nn.Conv2d: {t_torch:.4f} ms")
    try:
        import custom_conv
        w, b = conv.weight, conv.bias
        t_cuda = run_bench(lambda: custom_conv.custom_conv2d(x, w, b)[0], repeat=30)
        print(f"custom_conv2d:   {t_cuda:.4f} ms (speedup: {t_torch/t_cuda:.2f}x)")
    except ImportError:
        print("custom_conv not installed (pip install -e extension/)")